# 🤖 ML 모델 학습 → Pickle 직렬화 → GitHub 배포 파이프라인

**실행 순서:**
1. 라이브러리 설치 및 임포트
2. 간단한 ML 모델 학습 (Iris 분류)
3. Pickle 파일로 직렬화
4. GitHub API로 배포

## ⚙️ STEP 0. 설정값 입력 (필수)
> 아래 변수들을 본인의 GitHub 정보로 변경하세요.

In [ ]:
# ============================================================
#  ✏️  여기만 수정하세요
# ============================================================
GITHUB_TOKEN   = "ghp_XXXXXXXXXXXXXXXXXXXXXXXX"   # GitHub Personal Access Token
GITHUB_OWNER   = "your-username"                   # GitHub 사용자명 또는 조직명
GITHUB_REPO    = "your-repo-name"                  # 배포 대상 레포지토리 이름
GITHUB_BRANCH  = "main"                            # 브랜치명
REMOTE_PATH    = "models/iris_model.pkl"           # 레포 내 저장 경로
# ============================================================

LOCAL_PICKLE_PATH = "iris_model.pkl"               # 로컬 pickle 파일명

print("✅ 설정 완료")
print(f"   대상 : https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}/blob/{GITHUB_BRANCH}/{REMOTE_PATH}")

## 📦 STEP 1. 라이브러리 설치 및 임포트

In [ ]:
# 필요 라이브러리 설치 (scikit-learn, requests는 대부분 기본 설치됨)
!pip install scikit-learn requests --quiet

import pickle
import base64
import json
import requests
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("✅ 라이브러리 임포트 완료")

## 🌸 STEP 2. ML 모델 학습 (Iris 분류기)

In [ ]:
# ── 데이터 로드 ──────────────────────────────────────────────
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
target_names  = iris.target_names

print(f"데이터셋 크기  : {X.shape}")
print(f"클래스        : {target_names}")

# ── 학습 / 테스트 분리 ───────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── 모델 학습 ────────────────────────────────────────────────
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# ── 성능 평가 ────────────────────────────────────────────────
y_pred   = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n🎯 테스트 정확도 : {accuracy:.4f} ({accuracy*100:.2f}%)")
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

## 💾 STEP 3. Pickle 파일로 직렬화

In [ ]:
import os

# ── 저장할 메타데이터 포함한 패키지 구성 ────────────────────
model_package = {
    "model"         : model,
    "feature_names" : feature_names,
    "target_names"  : target_names,
    "accuracy"      : accuracy,
    "model_type"    : "RandomForestClassifier",
    "n_estimators"  : 100,
}

# ── Pickle 직렬화 ────────────────────────────────────────────
with open(LOCAL_PICKLE_PATH, "wb") as f:
    pickle.dump(model_package, f, protocol=pickle.HIGHEST_PROTOCOL)

file_size_kb = os.path.getsize(LOCAL_PICKLE_PATH) / 1024
print(f"✅ Pickle 저장 완료 : {LOCAL_PICKLE_PATH}")
print(f"   파일 크기 : {file_size_kb:.1f} KB")

# ── 역직렬화 검증 ────────────────────────────────────────────
with open(LOCAL_PICKLE_PATH, "rb") as f:
    loaded = pickle.load(f)

test_pred = loaded["model"].predict(X_test)
assert accuracy_score(y_test, test_pred) == accuracy, "역직렬화 검증 실패!"
print(f"✅ 역직렬화 검증 통과 (정확도 동일: {accuracy:.4f})")

## 🚀 STEP 4. GitHub API로 Pickle 파일 배포

> **사전 준비:** GitHub에서 PAT(Personal Access Token) 발급 필요  
> Settings → Developer settings → Personal access tokens → Tokens (classic)  
> 필요 권한: `repo` (전체)

In [ ]:
def deploy_to_github(
    local_path: str,
    remote_path: str,
    token: str,
    owner: str,
    repo: str,
    branch: str = "main",
    commit_message: str = None,
) -> dict:
    """
    로컬 파일을 GitHub 레포지토리에 업로드(Create 또는 Update)합니다.
    
    Returns:
        dict: GitHub API 응답
    """
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{remote_path}"
    headers = {
        "Authorization": f"token {token}",
        "Accept"       : "application/vnd.github.v3+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }

    # ── 파일을 Base64로 인코딩 ──────────────────────────────
    with open(local_path, "rb") as f:
        content_b64 = base64.b64encode(f.read()).decode("utf-8")

    # ── 기존 파일 SHA 확인 (Update 시 필요) ────────────────
    existing_sha = None
    check_resp = requests.get(api_url, headers=headers)
    if check_resp.status_code == 200:
        existing_sha = check_resp.json().get("sha")
        print(f"📄 기존 파일 감지 → Update 모드 (SHA: {existing_sha[:8]}...)")
    elif check_resp.status_code == 404:
        print("📄 신규 파일 → Create 모드")
    else:
        raise RuntimeError(f"GitHub API 오류: {check_resp.status_code} {check_resp.text}")

    # ── 커밋 메시지 자동 생성 ───────────────────────────────
    if commit_message is None:
        action = "Update" if existing_sha else "Add"
        commit_message = f"{action} ML model: {remote_path} (accuracy={accuracy:.4f})"

    # ── 업로드 페이로드 구성 ────────────────────────────────
    payload = {
        "message" : commit_message,
        "content" : content_b64,
        "branch"  : branch,
    }
    if existing_sha:
        payload["sha"] = existing_sha  # Update 시 필수

    # ── GitHub API PUT 요청 ─────────────────────────────────
    resp = requests.put(api_url, headers=headers, data=json.dumps(payload))

    return resp


print("✅ deploy_to_github() 함수 정의 완료")

In [ ]:
# ── 배포 실행 ────────────────────────────────────────────────
print(f"🚀 GitHub 배포 시작...")
print(f"   레포 : {GITHUB_OWNER}/{GITHUB_REPO}")
print(f"   경로 : {REMOTE_PATH}  (branch: {GITHUB_BRANCH})")
print()

response = deploy_to_github(
    local_path     = LOCAL_PICKLE_PATH,
    remote_path    = REMOTE_PATH,
    token          = GITHUB_TOKEN,
    owner          = GITHUB_OWNER,
    repo           = GITHUB_REPO,
    branch         = GITHUB_BRANCH,
)

# ── 결과 확인 ────────────────────────────────────────────────
if response.status_code in (200, 201):
    result = response.json()
    file_url = result["content"]["html_url"]
    commit_url = result["commit"]["html_url"]
    print(f"✅ 배포 성공!")
    print(f"   📁 파일  URL : {file_url}")
    print(f"   📝 커밋  URL : {commit_url}")
else:
    print(f"❌ 배포 실패: HTTP {response.status_code}")
    print(f"   응답: {response.json()}")

## 🔄 STEP 5. (선택) GitHub에서 모델 다운로드 및 추론 테스트

In [ ]:
def load_model_from_github(
    owner: str,
    repo: str,
    remote_path: str,
    branch: str = "main",
    token: str = None,
) -> dict:
    """
    GitHub 레포지토리에서 Pickle 모델을 다운로드하여 로드합니다.
    """
    import io
    
    # raw 파일 다운로드 URL
    raw_url = f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{remote_path}"
    
    headers = {}
    if token:  # Private 레포의 경우 토큰 필요
        headers["Authorization"] = f"token {token}"
    
    resp = requests.get(raw_url, headers=headers)
    resp.raise_for_status()
    
    return pickle.load(io.BytesIO(resp.content))


# ── GitHub에서 모델 로드 ─────────────────────────────────────
print("📥 GitHub에서 모델 다운로드 중...")
loaded_pkg = load_model_from_github(
    owner       = GITHUB_OWNER,
    repo        = GITHUB_REPO,
    remote_path = REMOTE_PATH,
    branch      = GITHUB_BRANCH,
    token       = GITHUB_TOKEN,   # Private 레포면 필요, Public이면 None으로 변경
)

print(f"✅ 모델 로드 성공!")
print(f"   모델 타입  : {loaded_pkg['model_type']}")
print(f"   저장 정확도: {loaded_pkg['accuracy']:.4f}")

# ── 추론 테스트 ──────────────────────────────────────────────
sample = np.array([[5.1, 3.5, 1.4, 0.2]])   # Iris-setosa 샘플
pred_class = loaded_pkg["model"].predict(sample)[0]
pred_proba = loaded_pkg["model"].predict_proba(sample)[0]

print(f"\n🌸 샘플 추론 결과")
print(f"   입력값  : {sample[0]}")
print(f"   예측 클래스: {loaded_pkg['target_names'][pred_class]}")
for cls, prob in zip(loaded_pkg["target_names"], pred_proba):
    bar = "█" * int(prob * 30)
    print(f"   {cls:<15} {prob:.4f}  {bar}")